In [31]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [32]:
load_dotenv()

True

In [33]:
# BaaseModel for structured output
class LlmResponse(BaseModel):
    
    review : str = Field(description='Give a review based on the prompt')
    marks : int = Field(description='Give marks based on the review', gt=0, le=10)

In [34]:
model = ChatOpenAI(model="gpt-4o-mini")
model_with_structure = model.with_structured_output(LlmResponse)

In [35]:
class EssayClass(TypedDict):
    essay : str
    feedback1 : str
    feedback2 : str 
    summary : str 
    marks : Annotated[list[int],  operator.add]
    total : float

In [36]:
graph = StateGraph(EssayClass)

In [37]:
def llm1(state : EssayClass):
    essay = state['essay']
    prompt = f'Give feedback for the following essay based on english language and coherence {essay} and give marks accordingly out of 10'
    
    feedback = model_with_structure.invoke(prompt).review 
    marks = model_with_structure.invoke(prompt).marks 
    return {'feedback1': feedback, 'marks':[marks]}
    

In [38]:
def llm2(state : EssayClass):
    essay = state['essay']
    prompt = f'Give feedback for the following essay based depth on the topic {essay} and give marks accordingly out of 10'
    
    feedback = model_with_structure.invoke(prompt).review 
    marks = model_with_structure.invoke(prompt).marks 
    return {'feedback2': feedback, 'marks':[marks]}
    

In [39]:
def summary(state : EssayClass):
    marks = sum(state['marks'])/len(state['marks'])
    return {'total': marks}

In [40]:
graph.add_node('llm1', llm1)
graph.add_node('llm2', llm2)
graph.add_node('summary', summary)

graph.add_edge(START, 'llm1')
graph.add_edge(START, 'llm2')
graph.add_edge('llm1', 'summary')
graph.add_edge('llm2', 'summary')
graph.add_edge('summary', END)
workflow = graph.compile()

In [41]:
essay = """ Growing up, I kept my head to the ground-literally.

I frequently searched the soil, collecting interesting rocks:, whether tucked beneath bushes, in forgotten cupholders, or even hidden in the tiny, useless front pockets of women’s jeans. I continue to collect rocks to this day because I believe that valuable lessons can be extracted from them. My collection is in no way expensive or high quality, but is instead rich in memories.

So what have I learned from my collection?

“Dragon egg”: When I was younger, my friends and I gathered fist-sized stones and pretended they were dragon eggs. We’d feed them magical soups (composed of shampoo, daisies, and glitter), which made the rocks sparkle in the sunshine. Out of sentimentality, I kept my dragon egg, because it reminds me never to lose my “sparkle” or creativity. Nowadays, I manifest this creativity through various outlets. I practice Minhwa, traditional Korean watercolor painting, with my mom bringing Korean folktale characters and scenery to life. I often play the piano, enjoying and treasuring music as a hobby. I even incorporate this “sparkle” into my outfits, adopting a wide range of fashion styles from Y2K, dark academia, to “Shrek-inspired.”. My creative hobbies are important for me to maintain balance-giving myself the rejuvenation and reflection needed to breathe fire into other aspects of my life.

Cracked pavement: My penchant for collecting shards of cracked pavement mirrors my love for tennis and sports, as both exemplify the beauty of resilience. Just as I’ve learned to appreciate the intricate textures and patterns in discarded concrete, I’ve honed my skills on the tennis court, eventually securing a spot on the varsity girls’ tennis team. These experiences have taught me that success is born from dedication and a willingness to find beauty in the journey, even amidst the cracks.

Sedimentary stones: Composed of diverse clasts-the rock equivalent of crumbs-sedimentary stones symbolize the intentional, purposeful role I’ve played in designing my academic journey. My passion for learning drives me to explore unique and specialized courses like Auto and Human Anatomy, which not only challenge my intellect but also resonate with real-world applications. And I don’t study to just do well on a test-I deeply contemplate how to apply course material in different aspects of education and life. For example, APUSH didn’t just teach me history, but also how to be a better leader. Analyzing documents throughout history has helped me successfully form innovative and effective policies leading SciOly and HOSA. Like sedimentary rocks, I collect “clasts,”, becoming more resilient, capable, and stronger along the way.

Patio stone: An ordinary stone taken from the bed of rocks on my patio left a small gap, from which a mystery tree now sprouts. In the seven years and four months since I first noticed it, I have been unable to identify what kind of tree it is. In that time, I have spent 2000 hours studying trees, rocks, and rivers and have dedicated 500 hours of that time learning how to identify 250+ different trees. However, I don’t mind the fact that I can’t identify the mystery tree; it reminds me that there is still more to discover, learn about, and study. Every glance at the tree (which quite literally “stumps” me) is a reminder to appreciate my growth and to continue to strive, even in rocky conditions.

In the pensive moments when I scan the ground for rocks, I find profound lessons hidden within the mundane. These unassuming fragments of the Earth’s history have been my unlikely mentors. Each rock is a reminder that growth and wisdom are found in the simplest of things, and as I reflect on these silent, unchanging witnesses of time, I am inspired to continue my journey.

Taking the next step, I plunge my hands into the deep, earthy soil once more. What kind of rock will I uncover? And where will it bring me next?

"""

In [42]:
response = workflow.invoke({'essay': essay})

In [43]:
print(response)

{'essay': ' Growing up, I kept my head to the ground-literally.\n\nI frequently searched the soil, collecting interesting rocks:, whether tucked beneath bushes, in forgotten cupholders, or even hidden in the tiny, useless front pockets of women’s jeans. I continue to collect rocks to this day because I believe that valuable lessons can be extracted from them. My collection is in no way expensive or high quality, but is instead rich in memories.\n\nSo what have I learned from my collection?\n\n“Dragon egg”: When I was younger, my friends and I gathered fist-sized stones and pretended they were dragon eggs. We’d feed them magical soups (composed of shampoo, daisies, and glitter), which made the rocks sparkle in the sunshine. Out of sentimentality, I kept my dragon egg, because it reminds me never to lose my “sparkle” or creativity. Nowadays, I manifest this creativity through various outlets. I practice Minhwa, traditional Korean watercolor painting, with my mom bringing Korean folktale 